<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_collection/Player_Match_Statistics_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FotMob API → Player Match Statistics → Excel

This Google Colab notebook collects **Premier League player match statistics directly from FotMob web API endpoints** and exports the final dataset to:

**`Player_Match_Statistics.xlsx`**

No previously downloaded FotMob JSON files are required.

## Workflow

1. Request each Premier League season from FotMob.
2. Get completed match IDs.
3. Request `matchDetails` for every completed match.
4. Extract one row per player appearance.
5. Keep all available FotMob player statistics.
6. Standardize the main variable names.
7. Create percentage and per-90 metrics.
8. Validate the final dataset.
9. Export everything to Excel.

> FotMob's web endpoints are unversioned and may change. Use a reasonable request delay and review the applicable terms before large-scale collection.

## 1. Install and import libraries

`openpyxl` is used to create and format the final Excel workbook.

In [1]:
!pip -q install openpyxl tqdm

In [2]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

## 2. Configuration

The Premier League FotMob league ID is **47**.

By default, the notebook collects:

- 2023/24
- 2024/25
- 2025/26

Set `MAX_MATCHES_PER_SEASON = None` for the full dataset.  
For a quick test, change it to a small number such as `5`.

In [3]:
LEAGUE_ID = 47
COUNTRY_CODE = "ENG"

SEASONS = [
    "2023/2024",
    "2024/2025",
    "2025/2026",
]

MAX_MATCHES_PER_SEASON = None

REQUEST_DELAY_MIN = 0.8
REQUEST_DELAY_MAX = 1.3

MAX_RETRIES = 4
REQUEST_TIMEOUT = 30

OUTPUT_FILE = Path("/content/Player_Match_Statistics.xlsx")

print("Seasons:", SEASONS)
print("Output :", OUTPUT_FILE)

Seasons: ['2023/2024', '2024/2025', '2025/2026']
Output : /content/Player_Match_Statistics.xlsx


## 3. Create the FotMob API session

The notebook first tries the `/api/data/` route used in the existing project notebooks and falls back to `/api/` if needed.

The helper also retries temporary errors and slows down when a `429 Too Many Requests` response is returned.

In [4]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.fotmob.com/",
})

API_BASES = [
    "https://www.fotmob.com/api/data",
    "https://www.fotmob.com/api",
]


def get_fotmob_json(endpoint, params=None, max_retries=MAX_RETRIES):
    '''
    Request JSON from a FotMob endpoint with retry handling.
    '''

    last_error = None

    for base_url in API_BASES:

        url = f"{base_url}/{endpoint}"

        for attempt in range(1, max_retries + 1):

            try:
                response = session.get(
                    url,
                    params=params,
                    timeout=REQUEST_TIMEOUT,
                )

                if response.status_code == 200:
                    return response.json()

                if response.status_code in {404, 410}:
                    last_error = (
                        f"HTTP {response.status_code} for {response.url}"
                    )
                    break

                if response.status_code == 429:
                    wait_seconds = 10 * attempt
                    print(
                        f"Rate limited (429). "
                        f"Waiting {wait_seconds} seconds..."
                    )
                    time.sleep(wait_seconds)
                    continue

                last_error = (
                    f"HTTP {response.status_code} for {response.url}"
                )

            except requests.RequestException as exc:
                last_error = str(exc)

            time.sleep(3 * attempt)

    raise RuntimeError(
        f"FotMob request failed. Last error: {last_error}"
    )

## 4. Test the API connection

This requests one Premier League season and verifies that fixture data is returned before the full collection starts.

In [5]:
test_season = SEASONS[0]

test_data = get_fotmob_json(
    "leagues",
    params={
        "id": LEAGUE_ID,
        "season": test_season,
        "ccode3": COUNTRY_CODE,
    },
)

test_matches = (
    test_data
    .get("fixtures", {})
    .get("allMatches", [])
)

print("API connection successful.")
print("Test season:", test_season)
print("Fixture records returned:", len(test_matches))

API connection successful.
Test season: 2023/2024
Fixture records returned: 381


## 5. Parse one `matchDetails` response

Each output row represents **one player appearance in one match**.

The parser combines:

- season and match information,
- player identity,
- team and opponent,
- home/away status,
- lineup and formation information,
- substitution information,
- every available FotMob player statistic.

In [6]:
def parse_match(match_data, season):
    '''
    Convert one FotMob matchDetails JSON response into
    one row per player appearance.
    '''

    general = match_data["general"]
    content = match_data["content"]

    match_id = int(general["matchId"])

    kickoff_utc = pd.to_datetime(
        general["matchTimeUTCDate"],
        utc=True,
    )

    gameweek = pd.to_numeric(
        general.get("matchRound"),
        errors="coerce",
    )

    home_team = general["homeTeam"]
    away_team = general["awayTeam"]

    home_team_id = home_team["id"]
    away_team_id = away_team["id"]

    position_map = {
        0: "GK",
        1: "DEF",
        2: "MID",
        3: "FWD",
    }

    lineup = content.get("lineup", {})
    lineup_lookup = {}

    for side in ["homeTeam", "awayTeam"]:

        team_lineup = lineup.get(side, {})
        formation = team_lineup.get("formation")

        for player in team_lineup.get("starters", []):

            performance = player.get("performance", {})

            sub_out_minute = None
            sub_reason = None

            for event in performance.get(
                "substitutionEvents",
                [],
            ):
                if event.get("type") == "subOut":
                    sub_out_minute = event.get("time")
                    sub_reason = event.get("reason")

            usual_position_id = player.get(
                "usualPlayingPositionId"
            )

            layout = player.get(
                "verticalLayout",
                {},
            )

            lineup_lookup[int(player["id"])] = {
                "Starts": True,
                "Position": position_map.get(
                    usual_position_id
                ),
                "Usual_Position_ID": usual_position_id,
                "Lineup_Position_ID": player.get(
                    "positionId"
                ),
                "Team_Formation": formation,
                "Is_Captain": player.get(
                    "isCaptain"
                ),
                "Starter_X": layout.get("x"),
                "Starter_Y": layout.get("y"),
                "Sub_In_Minute": None,
                "Sub_Out_Minute": sub_out_minute,
                "Sub_Reason": sub_reason,
            }

        for player in team_lineup.get("subs", []):

            performance = player.get("performance", {})

            sub_in_minute = None
            sub_reason = None

            for event in performance.get(
                "substitutionEvents",
                [],
            ):
                if event.get("type") == "subIn":
                    sub_in_minute = event.get("time")
                    sub_reason = event.get("reason")

            usual_position_id = player.get(
                "usualPlayingPositionId"
            )

            lineup_lookup[int(player["id"])] = {
                "Starts": False,
                "Position": position_map.get(
                    usual_position_id
                ),
                "Usual_Position_ID": usual_position_id,
                "Lineup_Position_ID": player.get(
                    "positionId"
                ),
                "Team_Formation": formation,
                "Is_Captain": player.get(
                    "isCaptain"
                ),
                "Starter_X": None,
                "Starter_Y": None,
                "Sub_In_Minute": sub_in_minute,
                "Sub_Out_Minute": None,
                "Sub_Reason": sub_reason,
            }

    player_stats = content.get(
        "playerStats",
        {},
    )

    rows = []

    for _, player in player_stats.items():

        if not player.get("stats"):
            continue

        player_id = int(player["id"])
        team_id = player.get("teamId")

        if team_id == home_team_id:

            opponent_id = away_team_id
            opponent = away_team["name"]
            home_away = "Home"

        elif team_id == away_team_id:

            opponent_id = home_team_id
            opponent = home_team["name"]
            home_away = "Away"

        else:

            opponent_id = None
            opponent = None
            home_away = None

        lineup_info = lineup_lookup.get(
            player_id,
            {},
        )

        row = {
            "Season": season,
            "Date": kickoff_utc.date(),
            "Kickoff_UTC": kickoff_utc,
            "Gameweek": gameweek,
            "Match_ID": match_id,

            "Player_ID": player_id,
            "Opta_ID": player.get("optaId"),
            "Player": player.get("name"),

            "Team_ID": team_id,
            "Team": player.get("teamName"),

            "Position": lineup_info.get(
                "Position"
            ),

            "Opponent_ID": opponent_id,
            "Opponent": opponent,
            "Home_Away": home_away,

            "Starts": lineup_info.get(
                "Starts"
            ),

            "Usual_Position_ID": lineup_info.get(
                "Usual_Position_ID",
                player.get("usualPosition"),
            ),

            "Lineup_Position_ID": lineup_info.get(
                "Lineup_Position_ID"
            ),

            "Team_Formation": lineup_info.get(
                "Team_Formation"
            ),

            "Is_Captain": lineup_info.get(
                "Is_Captain"
            ),

            "Starter_X": lineup_info.get(
                "Starter_X"
            ),

            "Starter_Y": lineup_info.get(
                "Starter_Y"
            ),

            "Sub_In_Minute": lineup_info.get(
                "Sub_In_Minute"
            ),

            "Sub_Out_Minute": lineup_info.get(
                "Sub_Out_Minute"
            ),

            "Sub_Reason": lineup_info.get(
                "Sub_Reason"
            ),

            "Is_Goalkeeper": player.get(
                "isGoalkeeper"
            ),

            "Shirt_Number": player.get(
                "shirtNumber"
            ),
        }

        for group in player.get(
            "stats",
            [],
        ):

            group_stats = group.get(
                "stats",
                {},
            )

            for stat_name, stat_info in group_stats.items():

                if not isinstance(
                    stat_info,
                    dict,
                ):
                    continue

                stat = stat_info.get(
                    "stat",
                    {},
                )

                if not isinstance(
                    stat,
                    dict,
                ):
                    continue

                value = stat.get("value")

                if (
                    stat_name not in row
                    or row[stat_name] is None
                ):
                    row[stat_name] = value

                total = stat.get("total")

                if total is not None:
                    row[
                        f"{stat_name} Total"
                    ] = total

                bonus = stat.get("bonus")

                if bonus is not None:
                    row[
                        f"{stat_name} Bonus"
                    ] = bonus

        rows.append(row)

    return pd.DataFrame(rows)

## 6. Collect all player match statistics directly from the API

For each season, the notebook:

1. requests the league fixture list,
2. keeps completed and non-canceled matches,
3. removes duplicate match IDs,
4. requests `matchDetails`,
5. parses player statistics immediately in memory.

The notebook also stores failed match IDs in a separate table for review.

In [7]:
all_player_matches = []
failed_matches = []
season_summary = []

for season in SEASONS:

    print("\n" + "=" * 72)
    print(f"SEASON: {season}")
    print("=" * 72)

    season_data = get_fotmob_json(
        "leagues",
        params={
            "id": LEAGUE_ID,
            "season": season,
            "ccode3": COUNTRY_CODE,
        },
    )

    all_matches = (
        season_data
        .get("fixtures", {})
        .get("allMatches", [])
    )

    finished_matches = []

    for match in all_matches:

        status = match.get(
            "status",
            {},
        )

        is_cancelled = (
            status.get("cancelled") is True
        )

        is_finished = (
            status.get("finished") is True
        )

        if (not is_cancelled) and is_finished:
            finished_matches.append(match)

    unique_matches = {
        match["id"]: match
        for match in finished_matches
    }

    finished_matches = list(
        unique_matches.values()
    )

    finished_matches = sorted(
        finished_matches,
        key=lambda x: x["id"],
    )

    if MAX_MATCHES_PER_SEASON is not None:
        finished_matches = finished_matches[
            :MAX_MATCHES_PER_SEASON
        ]

    print(
        "Fixture records returned :",
        f"{len(all_matches):,}",
    )

    print(
        "Completed unique matches :",
        f"{len(finished_matches):,}",
    )

    season_frames = []
    successful_matches = 0

    for match in tqdm(
        finished_matches,
        desc=f"Downloading {season}",
    ):

        match_id = match["id"]

        try:

            match_data = get_fotmob_json(
                "matchDetails",
                params={
                    "matchId": match_id
                },
            )

            match_df = parse_match(
                match_data,
                season=season,
            )

            if match_df.empty:

                failed_matches.append({
                    "Season": season,
                    "Match_ID": match_id,
                    "Reason": (
                        "No player statistics returned"
                    ),
                })

            else:

                season_frames.append(
                    match_df
                )

                successful_matches += 1

        except Exception as exc:

            failed_matches.append({
                "Season": season,
                "Match_ID": match_id,
                "Reason": str(exc),
            })

        time.sleep(
            random.uniform(
                REQUEST_DELAY_MIN,
                REQUEST_DELAY_MAX,
            )
        )

    if season_frames:

        season_player_df = pd.concat(
            season_frames,
            ignore_index=True,
            sort=False,
        )

        all_player_matches.append(
            season_player_df
        )

        player_rows = len(
            season_player_df
        )

    else:

        player_rows = 0

    season_summary.append({
        "Season": season,
        "Completed_Matches_Requested":
            len(finished_matches),
        "Successful_Matches":
            successful_matches,
        "Player_Appearance_Rows":
            player_rows,
    })


if not all_player_matches:
    raise RuntimeError(
        "No player match statistics were collected."
    )


player_matches_raw = pd.concat(
    all_player_matches,
    ignore_index=True,
    sort=False,
)


season_summary_df = pd.DataFrame(
    season_summary
)


failed_matches_df = pd.DataFrame(
    failed_matches
)


print("\n" + "=" * 72)
print("API COLLECTION SUMMARY")
print("=" * 72)

display(
    season_summary_df
)

print(
    "\nRaw player appearance rows:",
    f"{len(player_matches_raw):,}"
)

print(
    "Unique matches:",
    f"{player_matches_raw['Match_ID'].nunique():,}"
)

print(
    "Unique players:",
    f"{player_matches_raw['Player_ID'].nunique():,}"
)

print(
    "Failed matches:",
    f"{len(failed_matches_df):,}"
)

if not failed_matches_df.empty:
    display(
        failed_matches_df.head(20)
    )


SEASON: 2023/2024
Fixture records returned : 381
Completed unique matches : 380



SEASON: 2024/2025
Fixture records returned : 380
Completed unique matches : 380



SEASON: 2025/2026
Fixture records returned : 380
Completed unique matches : 380



API COLLECTION SUMMARY


,Season,Completed_Matches_Requested,Successful_Matches,Player_Appearance_Rows
0,2023/2024,380,380,11384
1,2024/2025,380,380,11567
2,2025/2026,380,380,11492



Raw player appearance rows: 34,443
Unique matches: 1,140
Unique players: 944
Failed matches: 0


## 7. Standardize the main statistic names

The API can return many statistics. This step renames the most important ones into analysis-friendly column names while leaving all other available FotMob columns in the dataset.

In [8]:
RENAME_MAP = {

    # General
    "FotMob rating": "Rating",
    "Minutes played": "Minutes",

    # Attack
    "Goals": "Goals",
    "Assists": "Assists",
    "Expected goals (xG)": "xG",
    "Expected goals on target (xGOT)": "xGOT",
    "Expected assists (xA)": "xA",
    "xG + xA": "xG_xA",
    "xG Non-penalty": "npxG",
    "Total shots": "Shots",
    "Shots on target": "Shots_On_Target",
    "Shots off target": "Shots_Off_Target",
    "Blocked shots": "Blocked_Shots",
    "Big chances missed": "Big_Chances_Missed",

    # Passing / creation
    "Accurate passes": "Accurate_Passes",
    "Accurate passes Total": "Pass_Attempts",
    "Chances created": "Chances_Created",
    "Big chances created": "Big_Chances_Created",
    "Passes into final third": "Passes_Final_Third",
    "Accurate long balls": "Accurate_Long_Balls",
    "Accurate long balls Total": "Long_Ball_Attempts",
    "Accurate crosses": "Accurate_Crosses",
    "Accurate crosses Total": "Cross_Attempts",

    # Possession / carrying
    "Touches": "Touches",
    "Touches in opposition box": "Touches_Opp_Box",
    "Successful dribbles": "Successful_Dribbles",
    "Successful dribbles Total": "Dribble_Attempts",
    "Dispossessed": "Dispossessed",
    "Offsides": "Offsides",

    # Defense
    "Defensive actions": "Defensive_Actions",
    "Tackles": "Tackles",
    "Tackles won": "Tackles_Won",
    "Last man tackle": "Last_Man_Tackle",
    "Blocks": "Blocks",
    "Clearances": "Clearances",
    "Interceptions": "Interceptions",
    "Recoveries": "Recoveries",
    "Dribbled past": "Dribbled_Past",

    # Duels
    "Ground duels won": "Ground_Duels_Won",
    "Ground duels won Total": "Ground_Duels_Total",
    "Aerial duels won": "Aerial_Duels_Won",
    "Aerial duels won Total": "Aerial_Duels_Total",
    "Duels won": "Duels_Won",
    "Duels lost": "Duels_Lost",
    "Was fouled": "Fouls_Won",
    "Fouls committed": "Fouls_Committed",

    # Goalkeeper
    "Saves": "Saves",
    "Goals conceded": "Goals_Conceded",
    "xGOT faced": "xGOT_Faced",
    "Goals prevented": "Goals_Prevented",
    "High claim": "High_Claims",
    "Sweeper actions": "Sweeper_Actions",
    "Punches": "Punches",
    "Throws": "Throws",
}


player_matches = (
    player_matches_raw
    .rename(
        columns=RENAME_MAP
    )
    .copy()
)

print(
    "Columns after standardization:",
    len(player_matches.columns)
)

Columns after standardization: 104


## 8. Create ratio metrics

These metrics are returned as decimal ratios.

Examples:

- `0.90` = 90%
- `0.75` = 75%

In [9]:
def safe_ratio(
    numerator,
    denominator,
):
    return np.where(
        denominator > 0,
        numerator / denominator,
        np.nan,
    )


RATIO_SPECS = [
    (
        "Accurate_Passes",
        "Pass_Attempts",
        "Pass_Accuracy",
    ),
    (
        "Accurate_Long_Balls",
        "Long_Ball_Attempts",
        "Long_Ball_Accuracy",
    ),
    (
        "Successful_Dribbles",
        "Dribble_Attempts",
        "Dribble_Success_Rate",
    ),
    (
        "Ground_Duels_Won",
        "Ground_Duels_Total",
        "Ground_Duel_Win_Rate",
    ),
    (
        "Aerial_Duels_Won",
        "Aerial_Duels_Total",
        "Aerial_Duel_Win_Rate",
    ),
]


for numerator_col, denominator_col, output_col in RATIO_SPECS:

    if {
        numerator_col,
        denominator_col,
    }.issubset(
        player_matches.columns
    ):

        numerator = pd.to_numeric(
            player_matches[
                numerator_col
            ],
            errors="coerce",
        )

        denominator = pd.to_numeric(
            player_matches[
                denominator_col
            ],
            errors="coerce",
        )

        player_matches[
            output_col
        ] = safe_ratio(
            numerator,
            denominator,
        )

## 9. Create per-90 metrics

Formula:

**Per 90 = Statistic ÷ Minutes × 90**

In [10]:
PER90_COLUMNS = [
    "Goals",
    "Assists",
    "xG",
    "xGOT",
    "xA",
    "Shots",
    "Shots_On_Target",
    "Chances_Created",
    "Touches_Opp_Box",
    "Successful_Dribbles",
    "Tackles",
    "Interceptions",
    "Recoveries",
    "Clearances",
    "Ground_Duels_Won",
    "Aerial_Duels_Won",
    "Fouls_Won",
    "Fouls_Committed",
]


if "Minutes" in player_matches.columns:

    minutes = pd.to_numeric(
        player_matches[
            "Minutes"
        ],
        errors="coerce",
    )

    for column in PER90_COLUMNS:

        if column not in player_matches.columns:
            continue

        values = pd.to_numeric(
            player_matches[
                column
            ],
            errors="coerce",
        )

        player_matches[
            f"{column}_Per90"
        ] = np.where(
            minutes > 0,
            values / minutes * 90,
            np.nan,
        )

## 10. Reorder the final columns

Match and player identification fields are moved to the beginning of the dataset. All remaining FotMob statistics follow them.

In [11]:
FIRST_COLUMNS = [
    "Season",
    "Date",
    "Kickoff_UTC",
    "Gameweek",
    "Match_ID",

    "Player_ID",
    "Opta_ID",
    "Player",

    "Team_ID",
    "Team",

    "Position",

    "Opponent_ID",
    "Opponent",

    "Home_Away",

    "Minutes",
    "Starts",

    "Rating",

    "Team_Formation",

    "Is_Captain",

    "Sub_In_Minute",
    "Sub_Out_Minute",

    "Is_Goalkeeper",
]


existing_first_columns = [
    column
    for column in FIRST_COLUMNS
    if column in player_matches.columns
]


other_columns = [
    column
    for column in player_matches.columns
    if column not in existing_first_columns
]


player_matches = player_matches[
    existing_first_columns
    + other_columns
]

## 11. Validate the final dataset

The intended primary key is:

**Season + Match_ID + Player_ID**

The validation checks row count, column count, unique matches, unique players, duplicate primary keys, and match counts by season.

In [12]:
duplicate_count = (
    player_matches
    .duplicated(
        subset=[
            "Season",
            "Match_ID",
            "Player_ID",
        ]
    )
    .sum()
)


print("=" * 72)
print("FINAL DATASET VALIDATION")
print("=" * 72)

print(
    "\nRows:",
    f"{len(player_matches):,}"
)

print(
    "Columns:",
    f"{len(player_matches.columns):,}"
)

print(
    "Unique matches:",
    f"{player_matches['Match_ID'].nunique():,}"
)

print(
    "Unique players:",
    f"{player_matches['Player_ID'].nunique():,}"
)

print(
    "Duplicate primary keys:",
    f"{duplicate_count:,}"
)


print("\nMatches by season:")

display(
    player_matches
    .groupby(
        "Season"
    )["Match_ID"]
    .nunique()
    .to_frame(
        "Matches"
    )
)


if "Position" in player_matches.columns:

    print(
        "\nPlayer appearances by position:"
    )

    display(
        player_matches[
            "Position"
        ]
        .value_counts(
            dropna=False
        )
        .to_frame(
            "Appearances"
        )
    )


display(
    player_matches.head(10)
)

FINAL DATASET VALIDATION

Rows: 34,443
Columns: 127
Unique matches: 1,140
Unique players: 944
Duplicate primary keys: 0

Matches by season:


,Matches
Season,
2023/2024,380
2024/2025,380
2025/2026,380



Player appearances by position:


,Appearances
Position,
MID,11214
DEF,11206
FWD,9706
GK,2313
None,4


,Season,Date,Kickoff_UTC,Gameweek,Match_ID,Player_ID,Opta_ID,Player,Team_ID,Team,...,Touches_Opp_Box_Per90,Successful_Dribbles_Per90,Tackles_Per90,Interceptions_Per90,Recoveries_Per90,Clearances_Per90,Ground_Duels_Won_Per90,Aerial_Duels_Won_Per90,Fouls_Won_Per90,Fouls_Committed_Per90
0,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,159833,58621,Kyle Walker,8456,Manchester City,...,0.0,0.000000,0.000000,0.000000,9.101124,1.011236,2.022472,1.011236,2.022472,1.011236
1,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,169200,61366,Kevin De Bruyne,8456,Manchester City,...,0.0,NaN,0.000000,0.000000,7.826087,0.000000,0.000000,0.000000,NaN,3.913043
2,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,201664,83283,Nathan Redmond,8191,Burnley,...,0.0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,NaN,0.000000
3,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,239219,91651,Mateo Kovacic,8456,Manchester City,...,0.0,1.343284,2.686567,0.000000,1.343284,0.000000,6.716418,0.000000,2.686567,1.343284
4,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,363364,121160,Ederson Moraes,8456,Manchester City,...,NaN,NaN,0.000000,0.000000,9.000000,0.000000,1.000000,NaN,1.000000,0.000000
5,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,411617,146941,Aymeric Laporte,8456,Manchester City,...,0.0,NaN,0.000000,0.000000,16.363636,8.181818,NaN,0.000000,NaN,0.000000
6,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,417068,126184,Nathan Aké,8456,Manchester City,...,0.0,NaN,3.417722,2.278481,5.696203,1.139241,4.556962,3.417722,1.139241,0.000000
7,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,484878,192290,Connor Roberts,8191,Burnley,...,0.0,NaN,2.000000,0.000000,3.000000,1.000000,2.000000,1.000000,NaN,1.000000
8,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,488139,165809,Bernardo Silva,8456,Manchester City,...,0.0,0.000000,1.000000,0.000000,4.000000,0.000000,1.000000,0.000000,NaN,0.000000
9,2023/2024,2023-08-11,2023-08-11 19:00:00+00:00,1,4193450,494597,172782,Josh Brownhill,8191,Burnley,...,0.0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,NaN,0.000000


## 12. Export to `Player_Match_Statistics.xlsx`

The workbook contains three sheets:

1. **Player Match Statistics** — the complete player-by-match dataset.
2. **Collection Summary** — API collection results by season.
3. **Failed Matches** — match IDs that could not be collected or parsed.

The main workbook is formatted with frozen headers, filters, bold column names, and readable widths.

In [13]:
export_df = player_matches.copy()


# Excel cannot store timezone-aware datetimes.
for column in export_df.select_dtypes(
    include=["datetimetz"]
).columns:

    export_df[column] = (
        export_df[column]
        .dt.tz_convert("UTC")
        .dt.tz_localize(None)
    )


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    export_df.to_excel(
        writer,
        sheet_name=(
            "Player Match Statistics"
        ),
        index=False,
    )

    season_summary_df.to_excel(
        writer,
        sheet_name=(
            "Collection Summary"
        ),
        index=False,
    )

    if failed_matches_df.empty:

        pd.DataFrame(
            columns=[
                "Season",
                "Match_ID",
                "Reason",
            ]
        ).to_excel(
            writer,
            sheet_name=(
                "Failed Matches"
            ),
            index=False,
        )

    else:

        failed_matches_df.to_excel(
            writer,
            sheet_name=(
                "Failed Matches"
            ),
            index=False,
        )


workbook = load_workbook(
    OUTPUT_FILE
)


for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"

    worksheet.auto_filter.ref = (
        worksheet.dimensions
    )

    for header_cell in worksheet[1]:

        header_cell.font = Font(
            bold=True
        )

        header_cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
        )

    for column_index in range(
        1,
        worksheet.max_column + 1,
    ):

        column_letter = (
            get_column_letter(
                column_index
            )
        )

        values = []

        for row_index in range(
            1,
            min(
                worksheet.max_row,
                500,
            ) + 1,
        ):

            value = worksheet.cell(
                row=row_index,
                column=column_index,
            ).value

            if value is not None:
                values.append(
                    len(str(value))
                )

        if values:
            width = min(
                max(values) + 2,
                35,
            )
        else:
            width = 12

        worksheet.column_dimensions[
            column_letter
        ].width = width


workbook.save(
    OUTPUT_FILE
)


print("=" * 72)
print("EXCEL EXPORT COMPLETE")
print("=" * 72)

print(
    "File:",
    OUTPUT_FILE
)

print(
    "Rows:",
    f"{len(export_df):,}"
)

print(
    "Columns:",
    f"{len(export_df.columns):,}"
)

EXCEL EXPORT COMPLETE
File: /content/Player_Match_Statistics.xlsx
Rows: 34,443
Columns: 127


## 13. Download the Excel file from Colab

Run this cell after the export is complete.

In [14]:
from google.colab import files

files.download(
    str(OUTPUT_FILE)
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>